# Milestone 8: Every version on every image size

Every pipeline version runs on 5 crops of your photo:

| Size | Megapixels |
|---|---|
| 640 × 428 | 0.27 |
| 1280 × 856 | 1.1 |
| 2560 × 1710 | 4.4 |
| 3840 × 2564 | 9.8 |
| 5796 × 3870 (full) | 22.4 |

Versions: **CPU** (M1), **naive** GPU (M2), **planar** 32 × 8 (M5), **shared** denoise (M6), **fused** (M7), plus the **copy in** and **copy out** times.

How it works:
- `benchmark_sizes.py` crops the Bayer data into `benchmark/size_WxH/data/`. Every crop starts at an even row and column, so it stays RGGB, the same idea as the one-pixel crop in Milestone 0.
- It runs the C++ and CUDA programs **inside** each crop folder, so they read `data/...` and write `output/...` exactly as before. No changes to the reading code were needed.
- Each program prints lines like `RESULT,fused,3.552`, which the Python script collects.
- For every size, the fused GPU output is checked against the CPU output with `compare_outputs`.

**What to look for:**
1. Does the GPU's advantage over the CPU **grow** with image size? A tiny image may not even give the GPU enough work to fill it.
2. **Throughput** (megapixels per millisecond) should rise for small images and then **level off** once the GPU is fully busy.
3. Do the copy times stay small compared with the kernels, or do they dominate?

The whole run takes about 1–2 minutes, mostly because of the CPU.

**Runtime must be T4 GPU.**

## Step 1: Connect Drive and go to the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/isp_project

## Step 2: Write the new files
- `cpu_benchmark_main.cpp`: the Milestone 1 pipeline, averaged over 3 runs after 1 warm-up.
- `gpu_benchmark_main.cu`: all four GPU versions plus copies, 1 warm-up + 10 averaged runs each. No new kernels; it reuses the ones from earlier milestones.
- `benchmark_sizes.py`: crops, runs, collects, and makes the table and chart.

In [ ]:
%%writefile cpu_benchmark_main.cpp
// cpu_benchmark_main.cpp
// Milestone 8: the CPU pipeline from Milestone 1, run several times so the
// time is averaged like the GPU versions. Prints one machine-readable line
//   RESULT,cpu,<milliseconds>
// which the Python script collects. Also saves output/cpu_output.bin.

#include <chrono>
#include <iostream>
#include <vector>

#include "cpu_stages.h"
#include "isp_io.h"

// The CPU is slow, so fewer timed runs than on the GPU (3 instead of 10).
const int CPU_TIMED_RUN_COUNT = 3;

int main() {
    ImageMetadata metadata;
    bool metadataOk = readMetadataFile("data/metadata.txt", metadata);
    if (!metadataOk) {
        return 1;
    }
    int pixelCount = metadata.imageWidth * metadata.imageHeight;
    std::vector<uint16_t> bayerPixels;
    bool bayerOk = readBayerFile("data/bayer.bin", pixelCount, bayerPixels);
    if (!bayerOk) {
        return 1;
    }

    // The vectors are created once, outside the loop. Run 0 is the warm-up:
    // it pays for the operating system setting up their memory (Milestone 1).
    std::vector<float> normalisedBayer;
    std::vector<float> rgbImage;
    std::vector<float> denoisedImage;
    std::vector<unsigned char> outputBytes;
    double totalMilliseconds = 0.0;

    for (int runNumber = 0; runNumber <= CPU_TIMED_RUN_COUNT; runNumber++) {
        std::chrono::steady_clock::time_point runStart = std::chrono::steady_clock::now();
        blackLevelCorrection(bayerPixels, metadata, normalisedBayer);
        demosaicBilinear(normalisedBayer, metadata.imageWidth, metadata.imageHeight, rgbImage);
        whiteBalance(rgbImage, metadata);
        denoiseGaussian3x3(rgbImage, metadata.imageWidth, metadata.imageHeight, denoisedImage);
        gammaAndConvertTo8Bit(denoisedImage, outputBytes);
        std::chrono::steady_clock::time_point runEnd = std::chrono::steady_clock::now();

        std::chrono::duration<double, std::milli> runTime = runEnd - runStart;
        if (runNumber > 0) {
            totalMilliseconds = totalMilliseconds + runTime.count();
        }
    }

    double averageMilliseconds = totalMilliseconds / CPU_TIMED_RUN_COUNT;
    std::cout << "RESULT,cpu," << averageMilliseconds << std::endl;

    bool writeOk = writeRgbFile("output/cpu_output.bin", outputBytes);
    if (!writeOk) {
        return 1;
    }
    return 0;
}


In [ ]:
%%writefile gpu_benchmark_main.cu
// gpu_benchmark_main.cu
// Milestone 8: times every GPU version on the image in ./data and prints
// machine-readable lines for the Python script:
//   RESULT,<version>,<milliseconds>
// Versions (all names from earlier milestones):
//   naive  - Milestone 2/4: interleaved RGB, 16x16 blocks, 5 kernels
//   planar - Milestone 5 config E: planar RGB, 32x8 blocks, 5 kernels
//   shared - Milestone 6: like planar, but shared-memory denoise
//   fused  - Milestone 7: one kernel for all 5 stages
//   copy_in / copy_out - CPU<->GPU copy times (the same for every version)
// Each version: 1 warm-up run + 10 averaged runs (Milestone 4 method).

#include <iostream>
#include <string>
#include <vector>

#include "cuda_helpers.h"
#include "gpu_buffers.h"
#include "gpu_fused_kernel.h"
#include "gpu_kernels.h"
#include "gpu_planar_kernels.h"
#include "gpu_shared_denoise.h"
#include "isp_io.h"

const int TIMED_RUN_COUNT = 10;
const int VERSION_NAIVE = 0;
const int VERSION_PLANAR = 1;
const int VERSION_SHARED = 2;
const int VERSION_FUSED = 3;
const int VERSION_COUNT = 4;

// Makes enough blocks of the given shape to cover the whole image.
static dim3 makeGridShape(const ImageMetadata& metadata, dim3 blockShape) {
    // dim3 stores unsigned numbers; convert to int so all the maths uses one type.
    int blockWidth = static_cast<int>(blockShape.x);
    int blockHeight = static_cast<int>(blockShape.y);
    int blocksAcross = (metadata.imageWidth + blockWidth - 1) / blockWidth;
    int blocksDown = (metadata.imageHeight + blockHeight - 1) / blockHeight;
    dim3 gridShape(blocksAcross, blocksDown);
    return gridShape;
}

static void launchNaive(DeviceBuffers& buffers, const ImageMetadata& metadata) {
    int imageWidth = metadata.imageWidth;
    int imageHeight = metadata.imageHeight;
    dim3 blockShape(16, 16);
    dim3 gridShape = makeGridShape(metadata, blockShape);

    blackLevelKernel<<<gridShape, blockShape>>>(buffers.deviceBayer, buffers.deviceNormalised,
                                                imageWidth, imageHeight,
                                                metadata.blackLevel, metadata.whiteLevel);
    checkCudaError(cudaGetLastError(), "launching blackLevelKernel");
    demosaicKernel<<<gridShape, blockShape>>>(buffers.deviceNormalised, buffers.deviceRgb,
                                              imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching demosaicKernel");
    whiteBalanceKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, imageWidth, imageHeight,
                                                  metadata.whiteBalanceRed,
                                                  metadata.whiteBalanceGreen,
                                                  metadata.whiteBalanceBlue);
    checkCudaError(cudaGetLastError(), "launching whiteBalanceKernel");
    denoiseKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, buffers.deviceDenoised,
                                             imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching denoiseKernel");
    gammaKernel<<<gridShape, blockShape>>>(buffers.deviceDenoised, buffers.deviceOutput,
                                           imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching gammaKernel");
}

// Planar 5-kernel pipeline. useSharedDenoise picks the Milestone 6 denoise.
static void launchPlanar(DeviceBuffers& buffers, const ImageMetadata& metadata, bool useSharedDenoise) {
    int imageWidth = metadata.imageWidth;
    int imageHeight = metadata.imageHeight;
    dim3 blockShape(32, 8);
    dim3 gridShape = makeGridShape(metadata, blockShape);

    blackLevelKernel<<<gridShape, blockShape>>>(buffers.deviceBayer, buffers.deviceNormalised,
                                                imageWidth, imageHeight,
                                                metadata.blackLevel, metadata.whiteLevel);
    checkCudaError(cudaGetLastError(), "launching blackLevelKernel");
    demosaicPlanarKernel<<<gridShape, blockShape>>>(buffers.deviceNormalised, buffers.deviceRgb,
                                                    imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching demosaicPlanarKernel");
    whiteBalancePlanarKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, imageWidth, imageHeight,
                                                        metadata.whiteBalanceRed,
                                                        metadata.whiteBalanceGreen,
                                                        metadata.whiteBalanceBlue);
    checkCudaError(cudaGetLastError(), "launching whiteBalancePlanarKernel");
    if (useSharedDenoise) {
        denoiseSharedKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, buffers.deviceDenoised,
                                                       imageWidth, imageHeight);
    } else {
        denoisePlanarKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, buffers.deviceDenoised,
                                                       imageWidth, imageHeight);
    }
    checkCudaError(cudaGetLastError(), "launching denoise");
    gammaPlanarKernel<<<gridShape, blockShape>>>(buffers.deviceDenoised, buffers.deviceOutput,
                                                 imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching gammaPlanarKernel");
}

static void launchFused(DeviceBuffers& buffers, const ImageMetadata& metadata) {
    dim3 blockShape(FUSED_TILE_WIDTH, FUSED_TILE_HEIGHT);
    dim3 gridShape = makeGridShape(metadata, blockShape);
    fusedPipelineKernel<<<gridShape, blockShape>>>(buffers.deviceBayer, buffers.deviceOutput,
                                                   metadata.imageWidth, metadata.imageHeight,
                                                   metadata.blackLevel, metadata.whiteLevel,
                                                   metadata.whiteBalanceRed,
                                                   metadata.whiteBalanceGreen,
                                                   metadata.whiteBalanceBlue);
    checkCudaError(cudaGetLastError(), "launching fusedPipelineKernel");
}

static void launchVersion(int version, DeviceBuffers& buffers, const ImageMetadata& metadata) {
    if (version == VERSION_NAIVE) {
        launchNaive(buffers, metadata);
    } else if (version == VERSION_PLANAR) {
        launchPlanar(buffers, metadata, false);
    } else if (version == VERSION_SHARED) {
        launchPlanar(buffers, metadata, true);
    } else {
        launchFused(buffers, metadata);
    }
}

static float averageMilliseconds(int version, DeviceBuffers& buffers, const ImageMetadata& metadata,
                                 cudaEvent_t startEvent, cudaEvent_t stopEvent) {
    float totalMilliseconds = 0.0f;
    for (int runNumber = 0; runNumber <= TIMED_RUN_COUNT; runNumber++) {
        checkCudaError(cudaEventRecord(startEvent), "recording start event");
        launchVersion(version, buffers, metadata);
        checkCudaError(cudaEventRecord(stopEvent), "recording stop event");
        checkCudaError(cudaEventSynchronize(stopEvent), "waiting for the run");
        float runMilliseconds = 0.0f;
        checkCudaError(cudaEventElapsedTime(&runMilliseconds, startEvent, stopEvent),
                       "reading run time");
        if (runNumber > 0) {
            totalMilliseconds = totalMilliseconds + runMilliseconds;
        }
    }
    return totalMilliseconds / TIMED_RUN_COUNT;
}

// Times one cudaMemcpy, averaged the same way as the kernels.
static float averageCopyMilliseconds(void* destination, const void* source, size_t byteCount,
                                     cudaMemcpyKind direction,
                                     cudaEvent_t startEvent, cudaEvent_t stopEvent) {
    float totalMilliseconds = 0.0f;
    for (int runNumber = 0; runNumber <= TIMED_RUN_COUNT; runNumber++) {
        checkCudaError(cudaEventRecord(startEvent), "recording start event");
        checkCudaError(cudaMemcpy(destination, source, byteCount, direction), "copying");
        checkCudaError(cudaEventRecord(stopEvent), "recording stop event");
        checkCudaError(cudaEventSynchronize(stopEvent), "waiting for the copy");
        float runMilliseconds = 0.0f;
        checkCudaError(cudaEventElapsedTime(&runMilliseconds, startEvent, stopEvent),
                       "reading copy time");
        if (runNumber > 0) {
            totalMilliseconds = totalMilliseconds + runMilliseconds;
        }
    }
    return totalMilliseconds / TIMED_RUN_COUNT;
}

int main() {
    ImageMetadata metadata;
    bool metadataOk = readMetadataFile("data/metadata.txt", metadata);
    if (!metadataOk) {
        return 1;
    }
    int pixelCount = metadata.imageWidth * metadata.imageHeight;
    std::vector<uint16_t> bayerPixels;
    bool bayerOk = readBayerFile("data/bayer.bin", pixelCount, bayerPixels);
    if (!bayerOk) {
        return 1;
    }

    DeviceBuffers buffers;
    allocateDeviceBuffers(buffers, pixelCount);
    std::vector<unsigned char> outputBytes(buffers.outputBytes);

    cudaEvent_t startEvent;
    cudaEvent_t stopEvent;
    checkCudaError(cudaEventCreate(&startEvent), "creating start event");
    checkCudaError(cudaEventCreate(&stopEvent), "creating stop event");

    // Copy in first (also leaves the Bayer data on the GPU for the kernels).
    float copyInMilliseconds = averageCopyMilliseconds(buffers.deviceBayer, bayerPixels.data(),
                                                       buffers.bayerBytes, cudaMemcpyHostToDevice,
                                                       startEvent, stopEvent);
    std::cout << "RESULT,copy_in," << copyInMilliseconds << std::endl;

    const char* versionNames[VERSION_COUNT] = {"naive", "planar", "shared", "fused"};
    for (int version = 0; version < VERSION_COUNT; version++) {
        float kernelMilliseconds = averageMilliseconds(version, buffers, metadata, startEvent, stopEvent);
        std::cout << "RESULT," << versionNames[version] << "," << kernelMilliseconds << std::endl;
    }

    // The last version run was "fused": time copying its output back, and save it.
    float copyOutMilliseconds = averageCopyMilliseconds(outputBytes.data(), buffers.deviceOutput,
                                                        buffers.outputBytes, cudaMemcpyDeviceToHost,
                                                        startEvent, stopEvent);
    std::cout << "RESULT,copy_out," << copyOutMilliseconds << std::endl;

    checkCudaError(cudaEventDestroy(startEvent), "destroying start event");
    checkCudaError(cudaEventDestroy(stopEvent), "destroying stop event");
    freeDeviceBuffers(buffers);

    bool writeOk = writeRgbFile("output/gpu_fused_output.bin", outputBytes);
    if (!writeOk) {
        return 1;
    }
    return 0;
}


In [ ]:
%%writefile benchmark_sizes.py
# benchmark_sizes.py
# Milestone 8: run every pipeline version on several image sizes.
#
# For each size it:
#   1. crops the full Bayer image (from data/) and saves it in its own folder,
#      benchmark/size_<width>x<height>/data/  (bayer.bin + metadata.txt)
#   2. runs the CPU benchmark and the GPU benchmark INSIDE that folder, so the
#      C++ programs read "data/..." and write "output/..." exactly as before
#   3. checks the fused GPU output against the CPU output with compare_outputs
# Then it saves benchmark/results.csv, prints a table, and draws a chart.

import os
import subprocess
import numpy
import matplotlib.pyplot as pyplot

# Sizes to test. Every width and height is EVEN, and every crop starts at an
# even row and column, so the top-left pixel is still red (RGGB) - the same
# reason as the one-pixel crop in Milestone 0.
cropSizes = [
    (640, 428),
    (1280, 856),
    (2560, 1710),
    (3840, 2564),
    (5796, 3870),
]

versionNames = ['cpu', 'naive', 'planar', 'shared', 'fused']


def readMetadata(metadataPath):
    metadata = {}
    for line in open(metadataPath):
        parts = line.split()
        metadata[parts[0]] = parts[1]
    return metadata


def makeEven(number):
    return number - (number % 2)


def writeCrop(fullBayer, fullMetadata, cropWidth, cropHeight):
    # Take the crop from the centre, so small crops still show the subject.
    fullHeight = fullBayer.shape[0]
    fullWidth = fullBayer.shape[1]
    startRow = makeEven((fullHeight - cropHeight) // 2)
    startColumn = makeEven((fullWidth - cropWidth) // 2)
    croppedBayer = fullBayer[startRow:startRow + cropHeight, startColumn:startColumn + cropWidth]

    sizeFolder = 'benchmark/size_' + str(cropWidth) + 'x' + str(cropHeight)
    os.makedirs(sizeFolder + '/data', exist_ok=True)
    os.makedirs(sizeFolder + '/output', exist_ok=True)

    numpy.ascontiguousarray(croppedBayer, dtype='<u2').tofile(sizeFolder + '/data/bayer.bin')

    # Same metadata as the full image, only width and height change.
    metadataFile = open(sizeFolder + '/data/metadata.txt', 'w')
    for fieldName in ['width', 'height', 'black_level', 'white_level', 'wb_red', 'wb_green', 'wb_blue']:
        fieldValue = fullMetadata[fieldName]
        if fieldName == 'width':
            fieldValue = str(cropWidth)
        if fieldName == 'height':
            fieldValue = str(cropHeight)
        metadataFile.write(fieldName + ' ' + fieldValue + '\n')
    metadataFile.close()
    return sizeFolder


def runProgram(programPath, workingFolder):
    # Runs a compiled program inside workingFolder and returns its printed text.
    completedRun = subprocess.run([programPath], cwd=workingFolder, capture_output=True, text=True)
    if completedRun.returncode != 0:
        print('ERROR running', programPath, 'in', workingFolder)
        print(completedRun.stdout)
        print(completedRun.stderr)
    return completedRun.stdout


def parseResults(programOutput, resultsForThisSize):
    # Lines look like "RESULT,fused,3.552". Everything else is ignored.
    for line in programOutput.splitlines():
        parts = line.strip().split(',')
        if len(parts) == 3 and parts[0] == 'RESULT':
            resultsForThisSize[parts[1]] = float(parts[2])


def main():
    fullMetadata = readMetadata('data/metadata.txt')
    fullWidth = int(fullMetadata['width'])
    fullHeight = int(fullMetadata['height'])
    fullBayer = numpy.fromfile('data/bayer.bin', dtype='<u2').reshape(fullHeight, fullWidth)

    allResults = []
    for cropSize in cropSizes:
        cropWidth = cropSize[0]
        cropHeight = cropSize[1]
        print('Running', cropWidth, 'x', cropHeight, '...')
        sizeFolder = writeCrop(fullBayer, fullMetadata, cropWidth, cropHeight)

        resultsForThisSize = {}
        resultsForThisSize['width'] = cropWidth
        resultsForThisSize['height'] = cropHeight
        resultsForThisSize['megapixels'] = cropWidth * cropHeight / 1000000.0

        cpuOutput = runProgram('/content/cpu_benchmark', sizeFolder)
        parseResults(cpuOutput, resultsForThisSize)
        gpuOutput = runProgram('/content/gpu_benchmark', sizeFolder)
        parseResults(gpuOutput, resultsForThisSize)

        compareCommand = ['/content/compare_outputs', 'output/cpu_output.bin', 'output/gpu_fused_output.bin']
        compareRun = subprocess.run(compareCommand, cwd=sizeFolder, capture_output=True, text=True)
        if 'RESULT: PASS' in compareRun.stdout:
            resultsForThisSize['fused_check'] = 'PASS'
        else:
            resultsForThisSize['fused_check'] = 'FAIL'

        allResults.append(resultsForThisSize)

    saveCsv(allResults)
    printTable(allResults)
    drawChart(allResults)


def saveCsv(allResults):
    columnNames = ['width', 'height', 'megapixels'] + versionNames + ['copy_in', 'copy_out', 'fused_check']
    csvFile = open('benchmark/results.csv', 'w')
    csvFile.write(','.join(columnNames) + '\n')
    for resultsForThisSize in allResults:
        rowValues = []
        for columnName in columnNames:
            rowValues.append(str(resultsForThisSize[columnName]))
        csvFile.write(','.join(rowValues) + '\n')
    csvFile.close()
    print('Saved benchmark/results.csv')


def printTable(allResults):
    print()
    print('Times in milliseconds (GPU columns = kernels only; copies listed separately)')
    header = '{:>11} {:>6} {:>9} {:>8} {:>8} {:>8} {:>8} {:>8} {:>9} {:>8} {:>6}'
    print(header.format('size', 'MP', 'cpu', 'naive', 'planar', 'shared', 'fused',
                        'copy in', 'copy out', 'cpu/fused', 'check'))
    row = '{:>11} {:>6.2f} {:>9.2f} {:>8.3f} {:>8.3f} {:>8.3f} {:>8.3f} {:>8.3f} {:>9.3f} {:>8.0f}x {:>6}'
    for resultsForThisSize in allResults:
        sizeLabel = str(resultsForThisSize['width']) + 'x' + str(resultsForThisSize['height'])
        speedUp = resultsForThisSize['cpu'] / resultsForThisSize['fused']
        print(row.format(sizeLabel, resultsForThisSize['megapixels'],
                         resultsForThisSize['cpu'], resultsForThisSize['naive'],
                         resultsForThisSize['planar'], resultsForThisSize['shared'],
                         resultsForThisSize['fused'], resultsForThisSize['copy_in'],
                         resultsForThisSize['copy_out'], speedUp,
                         resultsForThisSize['fused_check']))


def drawChart(allResults):
    # One colour AND one marker shape per version, so the lines can be told
    # apart even without colour (colour-blind readers, black-and-white prints).
    versionColours = {'cpu': '#2a78d6', 'naive': '#eb6834', 'planar': '#1baf7a',
                      'shared': '#eda100', 'fused': '#e87ba4'}
    versionMarkers = {'cpu': 'o', 'naive': 's', 'planar': '^', 'shared': 'D', 'fused': 'v'}
    versionLabels = {'cpu': 'CPU (C++)', 'naive': 'GPU naive (M2)', 'planar': 'GPU planar 32x8 (M5)',
                     'shared': 'GPU shared denoise (M6)', 'fused': 'GPU fused (M7)'}

    megapixelValues = []
    for resultsForThisSize in allResults:
        megapixelValues.append(resultsForThisSize['megapixels'])

    figure, (timeAxes, throughputAxes) = pyplot.subplots(1, 2, figsize=(13, 5))

    # Left: time vs image size. Both axes logarithmic, because the CPU is
    # hundreds of times slower and sizes span 0.3 to 22 megapixels.
    for versionName in versionNames:
        timeValues = []
        for resultsForThisSize in allResults:
            timeValues.append(resultsForThisSize[versionName])
        timeAxes.plot(megapixelValues, timeValues, color=versionColours[versionName],
                      marker=versionMarkers[versionName], markersize=8, linewidth=2,
                      label=versionLabels[versionName])
    timeAxes.set_xscale('log')
    timeAxes.set_yscale('log')
    timeAxes.set_xlabel('Image size (megapixels, log scale)')
    timeAxes.set_ylabel('Time per image (ms, log scale)')
    timeAxes.set_title('Processing time vs image size')
    timeAxes.grid(True, which='major', color='#e4e3df', linewidth=0.8)
    timeAxes.legend(frameon=False)

    # Right: throughput (megapixels processed per millisecond) for the GPU
    # versions. A flat line means the GPU is fully busy; a rising line means
    # small images do not give it enough work.
    for versionName in versionNames:
        if versionName == 'cpu':
            continue
        throughputValues = []
        for resultsForThisSize in allResults:
            throughputValues.append(resultsForThisSize['megapixels'] / resultsForThisSize[versionName])
        throughputAxes.plot(megapixelValues, throughputValues, color=versionColours[versionName],
                            marker=versionMarkers[versionName], markersize=8, linewidth=2,
                            label=versionLabels[versionName])
    throughputAxes.set_xscale('log')
    throughputAxes.set_ylim(bottom=0)
    throughputAxes.set_xlabel('Image size (megapixels, log scale)')
    throughputAxes.set_ylabel('Megapixels per millisecond (kernels only)')
    throughputAxes.set_title('GPU throughput vs image size')
    throughputAxes.grid(True, which='major', color='#e4e3df', linewidth=0.8)
    throughputAxes.legend(frameon=False)

    # Put a tick at each tested size, labelled in plain megapixels (0.3, 1.1 ...)
    # instead of the default powers of ten, which are hard to read.
    tickLabels = []
    for megapixels in megapixelValues:
        tickLabels.append('{:.1f}'.format(megapixels))

    for axes in [timeAxes, throughputAxes]:
        axes.set_xticks(megapixelValues)
        axes.set_xticklabels(tickLabels)
        axes.minorticks_off()
        axes.spines['top'].set_visible(False)
        axes.spines['right'].set_visible(False)

    figure.tight_layout()
    figure.savefig('benchmark/results_chart.png', dpi=130)
    print('Saved benchmark/results_chart.png')


main()


## Step 3: Compile the three programs
**Expected:** no output.

In [ ]:
!g++ -O2 -std=c++17 -o /content/cpu_benchmark cpu_benchmark_main.cpp cpu_pointwise_stages.cpp cpu_neighbourhood_stages.cpp isp_io.cpp
!nvcc -O2 -arch=sm_75 -o /content/gpu_benchmark gpu_benchmark_main.cu gpu_fused_kernel.cu gpu_shared_denoise.cu gpu_planar_kernels.cu gpu_buffers.cu gpu_pointwise_kernels.cu gpu_neighbourhood_kernels.cu cuda_helpers.cu isp_io.cpp
!g++ -O2 -std=c++17 -o /content/compare_outputs compare_outputs.cpp isp_io.cpp

## Step 4: Run the benchmark (CHECKPOINT 1)
**Expected:** `Running 640 x 428 ...` and so on for 5 sizes, then a table with one row per size, then `Saved benchmark/results_chart.png`. The **check** column should say **PASS** on every row.

**Paste the whole table to me.**

In [ ]:
!python benchmark_sizes.py

## Step 5: View the chart (CHECKPOINT 2)
**Left:** time per image on log-log axes. **Right:** GPU throughput (how many megapixels each GPU version processes per millisecond).

The chart and `results.csv` are saved in your Drive under `isp_project/benchmark/`. Just tell me it displayed. I'll read the shapes from your table.

In [ ]:
from IPython.display import Image as NotebookImage
from IPython.display import display
display(NotebookImage('benchmark/results_chart.png'))